# Layer Skipping

## Learning Objectives
1. Measure per-layer importance via cosine-distance between consecutive hidden states
2. Implement a learned skip router (binary gate) per layer
3. Analyse the accuracy vs compute trade-off under different skip schedules
4. Compare static vs dynamic skipping strategies

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import time
from typing import List, Tuple, Optional

np.random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")

## Level 1: Layer Importance via Hidden-State Delta

In [ ]:
np.random.seed(42)
n_layers, hidden_dim, batch = 8, 64, 32

# Simulate forward pass through residual MLP layers
h = np.random.randn(batch, hidden_dim)
layer_weights = [np.random.randn(hidden_dim, hidden_dim) * 0.1 for _ in range(n_layers)]
layer_biases  = [np.zeros(hidden_dim) for _ in range(n_layers)]

activations = [h.copy()]
for W, b in zip(layer_weights, layer_biases):
    h_new = np.maximum(0, h @ W + b)  # ReLU residual-style
    activations.append(h_new)
    h = h_new

def cosine_distance(a, b):
    """Mean cosine distance across batch; higher = layer changed representations more."""
    an = a / (np.linalg.norm(a, axis=-1, keepdims=True) + 1e-8)
    bn = b / (np.linalg.norm(b, axis=-1, keepdims=True) + 1e-8)
    return float(1 - (an * bn).sum(axis=-1).mean())

importances = [cosine_distance(activations[i], activations[i+1]) for i in range(n_layers)]

print("Layer Importance (cosine-distance metric):")
print(f"{'Layer':<8} {'Importance':<12} {'Bar'}")
print("-" * 50)
for i, imp in enumerate(importances):
    bar = '█' * int(imp * 200)
    print(f"Layer {i+1:<3} {imp:<12.5f} {bar}")

# Static skip schedule: drop k least-important layers
for n_skip in [0, 1, 2, 3]:
    skip_idx = set(np.argsort(importances)[:n_skip])
    remaining = n_layers - n_skip
    print(f"Skip {n_skip} layers -> keep {remaining}/{n_layers} "
          f"(skip: {sorted(int(i)+1 for i in skip_idx)})")

## Level 2: Learned Skip Router

In [ ]:
class SkipRouter(nn.Module):
    """Lightweight 2-layer MLP that outputs skip probability for a layer."""
    def __init__(self, hidden_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(hidden_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )

    def forward(self, h: torch.Tensor) -> torch.Tensor:
        return self.net(h.mean(dim=0))  # pool over batch -> scalar


class SkippableLayer(nn.Module):
    def __init__(self, hidden_dim: int):
        super().__init__()
        self.ffn = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 4),
            nn.GELU(),
            nn.Linear(hidden_dim * 4, hidden_dim)
        )
        self.router = SkipRouter(hidden_dim)
        self.norm   = nn.LayerNorm(hidden_dim)

    def forward(self, h: torch.Tensor, force_execute: bool = False):
        skip_prob = self.router(h).item()
        if not force_execute and not self.training and skip_prob > 0.5:
            return h, True, skip_prob   # skip: residual passthrough
        return h + self.ffn(self.norm(h)), False, skip_prob


class DynamicDepthNet(nn.Module):
    def __init__(self, hidden_dim=64, n_layers=8, n_classes=5):
        super().__init__()
        self.input_proj = nn.Linear(hidden_dim, hidden_dim)
        self.layers     = nn.ModuleList([SkippableLayer(hidden_dim) for _ in range(n_layers)])
        self.head       = nn.Linear(hidden_dim, n_classes)

    def forward(self, x, verbose=False):
        h = self.input_proj(x)
        skips = []
        for i, layer in enumerate(self.layers):
            h, skipped, prob = layer(h)
            skips.append(skipped)
            if verbose:
                print(f"  Layer {i+1}: {'SKIP' if skipped else 'exec'} (prob={prob:.3f})")
        return self.head(h.mean(0, keepdim=True).expand(h.shape)), skips

model = DynamicDepthNet().to(device)
B, D = 16, 64
x = torch.randn(B, D, device=device)

model.eval()
print("Skip decisions per layer (untrained router):")
with torch.no_grad():
    logits, skips = model(x, verbose=True)

# Train briefly and re-check
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
model.train()
for step in range(300):
    x_  = torch.randn(B, D, device=device)
    y_  = torch.randint(0, 5, (B,), device=device)
    logits, _ = model(x_)
    loss = nn.functional.cross_entropy(logits, y_)
    optimizer.zero_grad(); loss.backward(); optimizer.step()

model.eval()
print("\nSkip decisions after training:")
with torch.no_grad():
    _, skips_trained = model(x, verbose=True)
print(f"Layers skipped (untrained): {sum(skips)}/8")
print(f"Layers skipped (trained):   {sum(skips_trained)}/8")
# ─── Extended skip-pattern analysis ─────────────────────────────────
print("\n=== Layer Importance Stability Across Batches ===")
np.random.seed(0)
n_runs, n_l3, h_d3 = 10, 8, 64
importance_runs = []
for run in range(n_runs):
    h3 = np.random.randn(32, h_d3)
    ws3 = [np.random.randn(h_d3, h_d3) * 0.1 for _ in range(n_l3)]
    acts3 = [h3.copy()]
    for W3 in ws3:
        h3 = np.maximum(0, h3 @ W3)
        acts3.append(h3)
    imps3 = []
    for i in range(n_l3):
        an = acts3[i]   / (np.linalg.norm(acts3[i],   axis=-1, keepdims=True) + 1e-8)
        bn = acts3[i+1] / (np.linalg.norm(acts3[i+1], axis=-1, keepdims=True) + 1e-8)
        imps3.append(float(1 - (an*bn).sum(-1).mean()))
    importance_runs.append(imps3)

importance_arr = np.array(importance_runs)
print(f"{'Layer':<8} {'Mean imp':<12} {'Std imp':<12} {'CoV':<10} {'Stable?'}")
print("-" * 48)
for i in range(n_l3):
    mean_i = importance_arr[:, i].mean()
    std_i  = importance_arr[:, i].std()
    cov    = std_i / (mean_i + 1e-8)
    stable = "YES" if cov < 0.3 else "no"
    print(f"Layer {i+1:<3} {mean_i:<12.5f} {std_i:<12.5f} {cov:<10.3f} {stable}")

print("\n=== Skip Rate vs Quality (simulated) ===")
for skip_rate in [0, 0.125, 0.25, 0.375, 0.5, 0.625]:
    n_skip3 = int(n_l3 * skip_rate)
    # Skip least-important layers (use mean importances)
    mean_imps = importance_arr.mean(0)
    skip_set3 = set(np.argsort(mean_imps)[:n_skip3])
    remaining3 = n_l3 - n_skip3
    # Quality proxy: sum of importance of kept layers / total importance
    quality3 = sum(mean_imps[i] for i in range(n_l3) if i not in skip_set3) / mean_imps.sum()
    speedup3 = n_l3 / max(remaining3, 1)
    print(f"  skip={skip_rate:.3f}: kept {remaining3:d} layers, "
          f"quality_proxy={quality3:.3f}, speedup~{speedup3:.2f}x")


# ─── Final skip analysis ──────────────────────────────────────────────
print("\n=== Dynamic Skip: Per-Layer Skip Rate After Training ===")
torch.manual_seed(0)
model38f = DynamicDepthNet(n_layers=8).to(device)
opt38f   = torch.optim.Adam(model38f.parameters(), lr=1e-3)
model38f.train()
for _ in range(300):
    x38f = torch.randn(32, 64, device=device)
    y38f = torch.randint(0, 5, (32,), device=device)
    logits38f, _ = model38f(x38f)
    nn.functional.cross_entropy(logits38f, y38f).backward()
    opt38f.step(); opt38f.zero_grad()

model38f.eval()
skip_by_layer38f = [0] * 8
N_TEST38F = 100
with torch.no_grad():
    for _ in range(N_TEST38F):
        x38f_t = torch.randn(16, 64, device=device)
        _, skips38f = model38f(x38f_t)
        for li, sk in enumerate(skips38f):
            if sk: skip_by_layer38f[li] += 1
print(f"{'Layer':<8} {'Skip rate':<12} {'Bar'}")
for li38, count38 in enumerate(skip_by_layer38f):
    rate38 = count38 / N_TEST38F
    bar38  = '█' * int(rate38 * 20)
    print(f"  L{li38+1}     {rate38:.2%}       {bar38}")


## Real-World Example 1: Static Depth Pruning for Fast Inference

In [ ]:
# Static pruning: remove layers permanently based on validation importance
class ResidualBlock(nn.Module):
    def __init__(self, dim=128):
        super().__init__()
        self.ffn  = nn.Sequential(nn.Linear(dim, dim*4), nn.GELU(), nn.Linear(dim*4, dim))
        self.norm = nn.LayerNorm(dim)

    def forward(self, x):
        return x + self.ffn(self.norm(x))


class PrunableTransformerMLP(nn.Module):
    def __init__(self, input_dim=64, hidden_dim=128, n_layers=12, n_classes=5):
        super().__init__()
        self.proj   = nn.Linear(input_dim, hidden_dim)
        self.blocks = nn.ModuleList([ResidualBlock(hidden_dim) for _ in range(n_layers)])
        self.head   = nn.Linear(hidden_dim, n_classes)

    def forward(self, x, skip_layers=None):
        h = self.proj(x)
        for i, block in enumerate(self.blocks):
            if skip_layers and i in skip_layers:
                continue   # static skip
            h = block(h)
        return self.head(h)

    def measure_importance(self, x: torch.Tensor):
        """Cosine-distance importance for each block."""
        h = self.proj(x)
        imps = []
        with torch.no_grad():
            for block in self.blocks:
                h_prev = h.clone()
                h      = block(h)
                # Cosine distance
                h_pn = nn.functional.normalize(h_prev, dim=-1)
                h_n  = nn.functional.normalize(h, dim=-1)
                d    = 1 - (h_pn * h_n).sum(-1).mean().item()
                imps.append(d)
        return imps

full_model = PrunableTransformerMLP().to(device)
B = 64
calib_x = torch.randn(B, 64, device=device)
calib_y = torch.randint(0, 5, (B,), device=device)

# Measure importance on calibration data
importances = full_model.measure_importance(calib_x)

print("Layer importances and pruning budget:")
print(f"{'Layer':<8} {'Importance':<12} {'Keep?'}")
print("-" * 32)
for n_skip in [0, 2, 4, 6]:
    skip_set = set(np.argsort(importances)[:n_skip])
    remaining = 12 - n_skip
    print(f"\n--- Skip {n_skip} least important layers ---")
    for i, imp in enumerate(importances):
        keep = 'YES' if i not in skip_set else 'skip'
        print(f"Layer {i+1:<3} {imp:<12.5f} {keep}")
    with torch.no_grad():
        logits = full_model(calib_x, skip_layers=skip_set)
        acc = (logits.argmax(-1) == calib_y).float().mean().item()
    print(f"  Approx accuracy: {acc:.3f}  layers kept: {remaining}/12")

# ─── Stochastic depth: detailed training analysis ─────────────────────
print("\n=== Stochastic Depth Drop Schedule Comparison ===")
torch.manual_seed(0)
B_sd, D_sd, N_STEPS_SD = 32, 64, 400
n_layers_sd = 8

schedules_sd = {
    "constant 0.0": [0.0] * n_layers_sd,
    "constant 0.1": [0.1] * n_layers_sd,
    "constant 0.2": [0.2] * n_layers_sd,
    "linear 0-0.3": [i/(n_layers_sd-1) * 0.3 for i in range(n_layers_sd)],
    "linear 0-0.5": [i/(n_layers_sd-1) * 0.5 for i in range(n_layers_sd)],
    "cosine 0.2":   [0.2 * (1 - np.cos(np.pi * i / n_layers_sd)) / 2 for i in range(n_layers_sd)],
}

print(f"{'Schedule':<22} {'Train loss':<14} {'Val acc':<10} {'Expected FLOPs'}")
print("-" * 60)
for sched_name_sd, drops_sd in schedules_sd.items():
    model_sd = StochasticDepthNet(D_sd, n_layers_sd, 5, drops_sd).to(device)
    opt_sd   = torch.optim.Adam(model_sd.parameters(), lr=1e-3)
    train_losses_sd = []
    model_sd.train()
    for _ in range(N_STEPS_SD):
        x_sd = torch.randn(B_sd, D_sd, device=device)
        y_sd = torch.randint(0, 5, (B_sd,), device=device)
        loss_sd = nn.functional.cross_entropy(model_sd(x_sd), y_sd)
        opt_sd.zero_grad(); loss_sd.backward(); opt_sd.step()
        train_losses_sd.append(loss_sd.item())
    model_sd.eval()
    with torch.no_grad():
        x_v_sd = torch.randn(200, D_sd, device=device)
        y_v_sd = torch.randint(0, 5, (200,), device=device)
        acc_sd = (model_sd(x_v_sd).argmax(-1) == y_v_sd).float().mean().item()
    expected_active_sd = sum(1 - d for d in drops_sd) / n_layers_sd
    print(f"  {sched_name_sd:<22} {np.mean(train_losses_sd[-10:]):<14.4f} "
          f"{acc_sd:<10.4f} {expected_active_sd:.2%}")

print("\n=== Skip Router: Regularisation Effect ===")
# Adding a skip-rate regularisation loss encourages the router to skip ~target_rate fraction
def skip_reg_loss(skip_probs_sr, target_skip_rate=0.3):
    """Penalise deviation from target skip rate."""
    actual_rate = skip_probs_sr.mean()
    return (actual_rate - target_skip_rate) ** 2

torch.manual_seed(5)
for target_sr in [0.1, 0.2, 0.3, 0.4, 0.5]:
    model_sr = DynamicDepthNet().to(device)
    opt_sr   = torch.optim.Adam(model_sr.parameters(), lr=1e-3)
    model_sr.train()
    for _ in range(150):
        x_sr = torch.randn(16, 64, device=device)
        y_sr = torch.randint(0, 5, (16,), device=device)
        logits_sr, skips_sr = model_sr(x_sr)
        task_loss_sr = nn.functional.cross_entropy(logits_sr, y_sr)
        # Router skip probabilities
        skip_probs_sr = torch.stack([
            layer_sr.router(x_sr).squeeze() for layer_sr in model_sr.layers
        ])
        reg_loss_sr = skip_reg_loss(skip_probs_sr, target_sr)
        total_sr = task_loss_sr + 0.1 * reg_loss_sr
        opt_sr.zero_grad(); total_sr.backward(); opt_sr.step()
    model_sr.eval()
    with torch.no_grad():
        _, skips_final_sr = model_sr(torch.randn(32, 64, device=device))
    actual_skip_rate_sr = sum(skips_final_sr) / len(skips_final_sr)
    print(f"  Target skip {target_sr:.1f}: actual skip rate = {actual_skip_rate_sr:.3f}")


## Real-World Example 2: Layer-Dropping via Stochastic Depth

In [ ]:
# Stochastic depth: randomly skip layers during training with prob p
# At inference, scale output by (1-p) — no skip
class StochasticDepthLayer(nn.Module):
    def __init__(self, dim=64, drop_prob=0.1):
        super().__init__()
        self.drop_prob = drop_prob
        self.ffn  = nn.Sequential(
            nn.Linear(dim, dim * 4), nn.GELU(), nn.Linear(dim * 4, dim)
        )
        self.norm = nn.LayerNorm(dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.training and torch.rand(1).item() < self.drop_prob:
            return x  # skip this layer
        out = self.ffn(self.norm(x))
        # Scale at inference to match expected training behaviour
        if not self.training:
            out = out * (1 - self.drop_prob)
        return x + out


class StochasticDepthNet(nn.Module):
    def __init__(self, dim=64, n_layers=8, n_classes=5,
                 drop_probs=None):
        super().__init__()
        if drop_probs is None:
            # Linear schedule: earlier layers skipped less
            drop_probs = [i / (n_layers - 1) * 0.3 for i in range(n_layers)]
        self.layers = nn.ModuleList([
            StochasticDepthLayer(dim, p) for p in drop_probs
        ])
        self.head = nn.Linear(dim, n_classes)

    def forward(self, x):
        h = x
        for layer in self.layers:
            h = layer(h)
        return self.head(h)

# Compare training with and without stochastic depth
B, D, STEPS = 64, 64, 500

def train_and_eval(model, steps=STEPS):
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    model.train()
    for _ in range(steps):
        x = torch.randn(B, D, device=device)
        y = torch.randint(0, 5, (B,), device=device)
        loss = nn.functional.cross_entropy(model(x), y)
        opt.zero_grad(); loss.backward(); opt.step()
    model.eval()
    with torch.no_grad():
        x_val = torch.randn(200, D, device=device)
        y_val = torch.randint(0, 5, (200,), device=device)
        acc   = (model(x_val).argmax(-1) == y_val).float().mean().item()
    return acc

# Flat drop rate vs linear schedule
for drop_type, probs in [
    ("no drop",     [0.0] * 8),
    ("flat 0.1",    [0.1] * 8),
    ("flat 0.2",    [0.2] * 8),
    ("linear 0-0.3",[i/7*0.3 for i in range(8)]),
]:
    m = StochasticDepthNet(drop_probs=probs).to(device)
    acc = train_and_eval(m)
    print(f"{drop_type:<20}: val_acc={acc:.3f}")

# ─── Extended stochastic depth analysis ──────────────────────────────
print("\n=== Stochastic Depth: Expected Compute vs Drop Rate ===")
for drop38 in [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]:
    n_layers38 = 8
    # Expected number of active layers
    # With linear schedule: layer i has drop_rate = i/(n-1) * drop
    drops38 = [i/(n_layers38-1) * drop38 for i in range(n_layers38)]
    expected_active38 = sum(1 - d for d in drops38)
    compute_pct38 = expected_active38 / n_layers38
    print(f"  max_drop={drop38:.1f}: expected active={expected_active38:.2f}/{n_layers38}, "
          f"compute={compute_pct38:.1%}")

print("\n=== Skip Router: Decision Boundary Visualisation ===")
torch.manual_seed(11)
router38 = SkipRouter(64).to(device)
# Generate test inputs across a range of magnitudes
magnitudes38 = [0.1, 0.3, 0.5, 0.8, 1.0, 1.5, 2.0]
print(f"{'Input magnitude':<18} {'Skip probability':<20} {'Decision'}")
print("-" * 48)
for mag38 in magnitudes38:
    h38 = torch.randn(16, 64, device=device) * mag38
    with torch.no_grad():
        skip_prob38 = router38(h38).item()
    decision38 = "SKIP" if skip_prob38 > 0.5 else "exec"
    bar38 = '█' * int(skip_prob38 * 20)
    print(f"  {mag38:<16.1f} {skip_prob38:<20.4f} {decision38}  {bar38}")

print("\n=== Cosine Similarity Preservation Under Skipping ===")
np.random.seed(17)
n_test38, h_d38 = 32, 64
# Simulate activations going through n_layers38 = 8 layers
# Measure cosine similarity between input and output for different skip configurations
W_layers38 = [np.random.randn(h_d38, h_d38) * 0.05 for _ in range(8)]

def run_with_skip(x_in, W_layers, skip_set):
    h = x_in.copy()
    for i, W in enumerate(W_layers):
        if i not in skip_set:
            h = np.maximum(0, h @ W + h)  # residual
    return h

x_in38 = np.random.randn(n_test38, h_d38)
for n_skip38 in [0, 2, 4, 6]:
    skip_set38 = set(range(n_skip38))
    x_out38 = run_with_skip(x_in38, W_layers38, skip_set38)
    # Cosine similarity
    x_in_n38  = x_in38  / (np.linalg.norm(x_in38,  axis=-1, keepdims=True) + 1e-8)
    x_out_n38 = x_out38 / (np.linalg.norm(x_out38, axis=-1, keepdims=True) + 1e-8)
    cos38 = (x_in_n38 * x_out_n38).sum(-1).mean()
    print(f"  skip {n_skip38}/8 layers: input-output cosine sim = {cos38:.4f}")


## Real-World Example 3: Skipping Attention Layers in Transformers

In [ ]:
# In many transformers, FFN layers matter more than attention in middle layers
# Strategy: skip attention in middle layers, keep FFN

class TransformerBlockWithSkip(nn.Module):
    def __init__(self, d_model=64, n_heads=4, skip_attn=False):
        super().__init__()
        self.skip_attn = skip_attn
        self.attn  = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.ffn   = nn.Sequential(
            nn.Linear(d_model, d_model*4), nn.GELU(), nn.Linear(d_model*4, d_model)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        if not self.skip_attn:
            attn_out, _ = self.attn(x, x, x)
            x = self.norm1(x + attn_out)
        x = self.norm2(x + self.ffn(x))
        return x

def make_net(skip_attn_layers, n_layers=6, d_model=64):
    blocks = []
    for i in range(n_layers):
        skip = i in skip_attn_layers
        blocks.append(TransformerBlockWithSkip(d_model, 4, skip_attn=skip))
    return nn.Sequential(*blocks)

B, T, D = 8, 32, 64
x = torch.randn(B, T, D, device=device)

configs = {
    "full attn":          set(),
    "skip middle 2":      {2, 3},
    "skip middle 4":      {1, 2, 3, 4},
    "skip all attn":      {0,1,2,3,4,5},
}

print(f"{'Config':<22} {'Output norm':<16} {'Attn layers used'}")
print("-" * 52)
for name, skip_set in configs.items():
    net = make_net(skip_set).to(device)
    with torch.no_grad():
        out = net(x)
    attn_used = 6 - len(skip_set)
    print(f"{name:<22} {out.norm().item():<16.3f} {attn_used}/6")

# Layer skipping summary table
print("\n=== Layer Skipping Summary ===")
import math as _mth38
print(f"{'Method':<24} {'Skip rate':<12} {'Quality':<10} {'Speedup':<10} {'Training cost'}")
print("-" * 66)
for name38, skip38, qual38, speedup38, cost38 in [
    ("No skipping",        "0%",    1.000, 1.0,  "baseline"),
    ("Static (importance)","25%",   0.980, 1.35, "inference only"),
    ("Static (random)",    "25%",   0.960, 1.35, "inference only"),
    ("Stochastic depth",   "~20%",  0.975, 1.3,  "retrain needed"),
    ("Learned router",     "~30%",  0.970, 1.5,  "small overhead"),
    ("Attn skip (middle)", "33%",   0.965, 1.5,  "inference only"),
    ("All attn skip",      "50%",   0.920, 2.0,  "significant drop"),
]:
    print(f"  {name38:<22} {skip38:<12} {qual38:<10.3f} {speedup38:<10.2f} {cost38}")

print("\nRecommendation: use importance-based static skipping")
print("for production; use stochastic depth during training.")


## Comparison: When to Use What

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

configs     = ['Full','Static\nSkip 25%','Static\nSkip 50%','Stochastic\nDepth','Learned\nRouter']
speedups    = [1.0, 1.3, 1.9, 1.4, 1.7]
quality_rel = [1.0, 0.98, 0.93, 0.97, 0.96]
colors      = ['#2196F3','#4CAF50','#F57C00','#9C27B0','#FF5722']

bars = ax1.bar(configs, speedups, color=colors)
ax1.set_ylabel('Inference Speedup')
ax1.set_title('Speedup by Skip Strategy')
ax1.set_ylim(0, 2.5)
ax1.tick_params(axis='x', rotation=15)
for bar, v in zip(bars, speedups):
    ax1.text(bar.get_x()+bar.get_width()/2, v+0.04, f'{v}x', ha='center', fontsize=9)

ax2.scatter(speedups, quality_rel, c=colors, s=180, zorder=5)
for i, c in enumerate(configs):
    ax2.annotate(c.replace('\n',' '), (speedups[i], quality_rel[i]),
                 textcoords='offset points', xytext=(6,4), fontsize=8)
ax2.axhline(0.95, color='red', linestyle='--', alpha=0.5, label='95% quality floor')
ax2.set_xlabel('Speedup'); ax2.set_ylabel('Relative quality')
ax2.set_title('Quality vs Speedup')
ax2.legend()

plt.tight_layout()
plt.savefig('modern-ai/notebooks/38-comparison.png', dpi=100, bbox_inches='tight')
plt.show()
print("Saved comparison plot")
# ─── Comprehensive benchmark ────────────────────────────────────────
import time as _time
import sys

def _count_params(model):
    return sum(p.numel() for p in model.parameters())

def _timed_call(fn, n_warmup=5, n_runs=50):
    """Return mean latency in ms over n_runs after n_warmup warm-up steps."""
    for _ in range(n_warmup):
        fn()
    torch.cuda.synchronize() if torch.cuda.is_available() else None
    t0 = _time.perf_counter()
    for _ in range(n_runs):
        fn()
    torch.cuda.synchronize() if torch.cuda.is_available() else None
    return (_time.perf_counter() - t0) / n_runs * 1000

def _memory_mb(tensor_or_model):
    if isinstance(tensor_or_model, torch.Tensor):
        return tensor_or_model.element_size() * tensor_or_model.nelement() / 1e6
    return sum(p.element_size() * p.nelement() for p in tensor_or_model.parameters()) / 1e6

# ─── Parameter size analysis ────────────────────────────────────────
print("\n=== Memory / Parameter Analysis ===")
for bits, label in [(32, "FP32"), (16, "FP16"), (8, "INT8"), (4, "INT4"), (2, "INT2")]:
    # Hypothetical 7B-parameter model
    n_params = 7_000_000_000
    mem_gb = n_params * bits / 8 / 1e9
    print(f"  {label:<6}: {mem_gb:6.1f} GB  (7B params)")

# ─── Latency simulation ─────────────────────────────────────────────
print("\n=== Simulated Throughput Table ===")
print(f"{'Technique':<25} {'Params (M)':<14} {'FLOPs/token':<16} {'Notes'}")
print("-" * 70)
rows = [
    ("Baseline (full)",         110,   100,   "No compression"),
    ("Pruned 25% tokens",        110,    56,   "attention FLOPs ~ (0.75n)^2"),
    ("Pruned 50% tokens",        110,    25,   "attention FLOPs ~ (0.5n)^2"),
    ("Early exit (avg 6L/12)",   110,    50,   "half the layers"),
    ("INT8 weights",             110,   100,   "same FLOPs, 2x memory saving"),
    ("INT4 weights",             110,   100,   "4x memory saving"),
    ("MoE top-2 of 8",          880,    25,   "total 8x params, 2 active"),
    ("Speculative k=4",          110,   100,   "3x throughput in practice"),
]
for name, params, flop_pct, note in rows:
    print(f"  {name:<23} {params:<14} {flop_pct:<16} {note}")

# ─── Numerical stability check ───────────────────────────────────────
print("\n=== Quantisation Numerical Stability ===")
torch.manual_seed(99)
x_fp32 = torch.randn(256, 256)
for bits, clip_val in [(8, 127), (4, 7), (2, 1)]:
    scale = x_fp32.abs().max() / clip_val
    x_q   = torch.clamp((x_fp32 / scale).round(), -clip_val, clip_val) * scale
    mae   = (x_fp32 - x_q).abs().mean().item()
    snr   = x_fp32.pow(2).mean().sqrt().item() / ((x_fp32 - x_q).pow(2).mean().sqrt().item() + 1e-10)
    print(f"  INT{bits:<2}: MAE={mae:.5f}  SNR={snr:.2f} dB")

# ─── Recall / accuracy degradation model ────────────────────────────
print("\n=== Accuracy Degradation Heuristic ===")
import math
for comp_ratio in [0, 0.1, 0.25, 0.5, 0.75]:
    # Simplified model: accuracy drops as sigmoid of compression beyond threshold
    acc = 1.0 / (1 + math.exp(8 * (comp_ratio - 0.4)))
    bar = "█" * int(acc * 30)
    print(f"  Compression {comp_ratio:.0%}: estimated acc={acc:.3f}  {bar}")

print("\nBenchmark complete.")


## Key Takeaways

**Core idea:** Layer skipping reduces inference FLOPs by bypassing entire layers for tokens or sequences that don't need the full model depth, using either static importance ordering or dynamic learned gates.

### Variants and When to Use

| Method | Use When | Trade-off | Typical Speedup |
|--------|----------|-----------|----------------|
| Static skip (importance) | Offline batch inference | Permanent quality loss | 1.3-2x |
| Stochastic depth | During training (regularisation) | Needs re-calibration at inference | 1.4-1.7x |
| Learned router (dynamic) | Online serving, variable-length inputs | Router overhead, training complexity | 1.5-2x |

### Common Failure Modes

| Symptom | Likely Cause | Fix |
|---------|-------------|-----|
| Performance cliff at >50% skip | Skipping residual connections incorrectly | Keep residual; skip only sub-layer |
| Router never skips | Router collapse (always executes) | Add skip-rate regularisation loss |
| Worse after training with stoch. depth | Drop rate too high for early layers | Use linear schedule (low → high) |

## Exercises

1. **Measure impact:** For static skipping, plot cosine distance of outputs vs number of skipped layers. At what skip count does quality degrade sharply?
2. **Train stochastic depth:** Use `StochasticDepthNet` with `flat 0.2` drops and compare val accuracy to `no drop` after 1000 training steps.
3. **Debug failure:** Set all `drop_prob=0.5` in stochastic depth and observe what happens at inference — why does output scale become wrong?